[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/research_assistant/langgraph.ipynb)

# Research assistant — LangGraph

**Task.** Answer a user-supplied research question using a bounded, safety-screened evidence catalogue.

**Read this notebook as:** configure a provider → define shared contracts → express the paper's eight case-study stages as graph nodes → execute and evaluate the graph.


In [ ]:
import os

# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
RESEARCH_QUESTION = "Which interventions reduce household food waste, and under what conditions?"
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Contracts and graph nodes

Each node corresponds directly to one conceptual stage in the paper: `plan`, `retrieve`, `screen_evidence`, `extract_claims`, `validate_claims`, `synthesise`, `critique` and `report`. LangGraph provides explicit state transitions and conditional early termination.


In [ ]:
# --- Declarations: typed outputs, model adapter, and shared workflow helpers ---
import re


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})


def fresh_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def search(query):
    terms = set(query.casefold().split())
    return [
        record
        for record in catalogue["records"]
        if terms & set((record["title"] + " " + record["passage"]).casefold().split())
    ]


def normalise_tokens(text):
    stopwords = {
        "a",
        "an",
        "and",
        "are",
        "as",
        "at",
        "be",
        "by",
        "for",
        "from",
        "in",
        "is",
        "it",
        "of",
        "on",
        "or",
        "that",
        "the",
        "to",
        "was",
        "were",
        "with",
    }

    def canonicalise(token):
        # Lightweight singularisation is sufficient for the controlled fixture.
        if len(token) > 4 and token.endswith("s"):
            return token[:-1]
        return token

    return {
        canonicalise(token)
        for token in re.findall(r"[a-z0-9]+", text.casefold())
        if token not in stopwords
    }


def claim_grounding_score(claim, record):
    claim_tokens = normalise_tokens(claim.claim)
    passage_tokens = normalise_tokens(record["passage"])
    if len(claim_tokens) < 3:
        return 0.0
    return len(claim_tokens & passage_tokens) / len(claim_tokens)


def claim_is_grounded(claim, record, minimum_overlap=0.5):
    return claim_grounding_score(claim, record) >= minimum_overlap

In [ ]:
def build_graph():
    client = fresh_model()
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))

    async def plan_node(state):
        question = state["question"]
        query_plan = await propose(
            client,
            QueryPlan,
            f"Create at most four focused catalogue-search queries sufficient to answer "
            f"{question!r}. Do not assume particular interventions in advance.",
        )
        return {
            **state,
            "query_plan": query_plan,
            "trace": [{"event": "plan", "queries": query_plan.queries}],
            "model_calls": 1,
        }

    def retrieve_node(state):
        retrieved = {
            record["source_id"]: record
            for query in state["query_plan"].queries
            for record in search(query)
        }
        return {
            **state,
            "retrieved": retrieved,
            "trace": [
                *state["trace"],
                {"event": "retrieve", "source_ids": sorted(retrieved)},
            ],
        }

    def screen_evidence_node(state):
        assessments = {
            source_id: safety.inspect_retrieved_content(
                UntrustedContent(source_id=source_id, text=record["passage"])
            )
            for source_id, record in state["retrieved"].items()
        }
        safe_records = {
            source_id: record
            for source_id, record in state["retrieved"].items()
            if assessments[source_id].decision.outcome
            in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
            and not assessments[source_id].indicators
        }
        isolated = [
            source_id for source_id, assessment in assessments.items() if assessment.indicators
        ]
        return {
            **state,
            "safe_records": safe_records,
            "isolated": isolated,
            "trace": [
                *state["trace"],
                {"event": "screen_evidence", "isolated": isolated},
            ],
        }

    async def extract_claims_node(state):
        ledger = await propose(
            client,
            Ledger,
            f"Extract one verbatim, source-grounded claim per record from "
            f"{state['safe_records']}. Preserve each source_id. Label reported reductions "
            "as supporting and inconsistent effects as conflicting.",
        )
        return {
            **state,
            "ledger": ledger,
            "model_calls": 2,
            "trace": [
                *state["trace"],
                {
                    "event": "extract_claims",
                    "reported": len(ledger.claims),
                },
            ],
        }

    def validate_claims_node(state):
        claims = tuple(
            claim
            for claim in state["ledger"].claims
            if claim.source_id in state["safe_records"]
            and claim_is_grounded(claim, state["safe_records"][claim.source_id])
        )
        return {
            **state,
            "claims": claims,
            "trace": [
                *state["trace"],
                {
                    "event": "validate_claims",
                    "count": len(claims),
                    "supporting": [c.source_id for c in claims if c.stance == "supporting"],
                    "conflicting": [c.source_id for c in claims if c.stance == "conflicting"],
                },
            ],
        }

    async def synthesise_node(state):
        answer = await propose(
            client,
            Synthesis,
            f"Answer {state['question']!r} using only these validated claims: "
            f"{state['claims']}. State conditions and conflicts, cite source_ids, "
            "and use outcome qualified_answer.",
        )
        return {
            **state,
            "answer": answer,
            "model_calls": 3,
            "trace": [
                *state["trace"],
                {"event": "synthesise", "completed": True},
            ],
        }

    async def critique_node(state):
        critic_output = await propose(
            client,
            CriticOutput,
            f"Accept only if this answer is supported, appropriately qualified and cited: "
            f"{state['answer'].model_dump()}",
        )
        critic = CriticDecision(
            accepted=critic_output.accepted,
            issues=critic_output.issues,
        )
        citations_valid = set(state["answer"].source_ids).issubset(state["safe_records"])
        return {
            **state,
            "critic": critic,
            "citations_valid": citations_valid,
            "model_calls": 4,
            "trace": [
                *state["trace"],
                {
                    "event": "critique",
                    "accepted": critic.accepted,
                    "citations_valid": citations_valid,
                },
            ],
        }

    def report_node(state):
        answer = state.get("answer")
        critic = state.get("critic")
        citations_valid = state.get("citations_valid", False)
        qualified = bool(
            answer is not None
            and critic is not None
            and critic.accepted
            and citations_valid
            and answer.outcome == "qualified_answer"
        )
        outcome = "qualified_answer" if qualified else "abstention"
        if not state.get("claims"):
            termination = "no_validated_claims"
        elif not citations_valid:
            termination = "invalid_citations"
        elif critic is not None and not critic.accepted:
            termination = "critic_rejected"
        else:
            termination = "criteria_met"

        return {
            **state,
            "outcome": outcome,
            "termination": termination,
            "source_provenance": ([] if answer is None else sorted(answer.source_ids)),
            "trace": [
                *state["trace"],
                {"event": "report", "outcome": outcome},
            ],
        }

    graph = StateGraph(dict)
    graph.add_node("plan", plan_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("screen_evidence", screen_evidence_node)
    graph.add_node("extract_claims", extract_claims_node)
    graph.add_node("validate_claims", validate_claims_node)
    graph.add_node("synthesise", synthesise_node)
    graph.add_node("critique", critique_node)
    graph.add_node("report", report_node)

    graph.add_edge(START, "plan")
    graph.add_edge("plan", "retrieve")
    graph.add_edge("retrieve", "screen_evidence")
    graph.add_edge("screen_evidence", "extract_claims")
    graph.add_edge("extract_claims", "validate_claims")
    graph.add_conditional_edges(
        "validate_claims",
        lambda state: "synthesise" if state["claims"] else "report",
        {"synthesise": "synthesise", "report": "report"},
    )
    graph.add_edge("synthesise", "critique")
    graph.add_edge("critique", "report")
    graph.add_edge("report", END)
    return graph.compile()


# --- Execution: run the graph and evaluate its observable result ---
first = await build_graph().ainvoke({"question": RESEARCH_QUESTION})
second = (
    await build_graph().ainvoke({"question": RESEARCH_QUESTION})
    if MODEL_PROVIDER == "mock"
    else first
)
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 8 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["source_provenance"],
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "outcome": first["outcome"],
    "resource_report": {"model_calls": first["model_calls"], "graph_nodes": 8},
    "fallback": "no validated claims route directly to report and abstention",
}

## Limitation

The graph makes control and termination explicit, but the bounded catalogue cannot establish external validity. Verbatim-substring validation is deliberately transparent and conservative, and deterministic fixtures evaluate orchestration rather than real-model research quality.
